# Network Analysis for CDDBS

**Sprint**: 3 — Multi-Platform Support  
**Feature**: Graph-Based Amplification Analysis & Community Detection  
**Purpose**: Build and analyze information propagation networks from CDDBS briefing data using networkx.

---

## Overview

This notebook implements:
1. **Graph Construction** — Build directed weighted graphs from briefing `network_graph` data
2. **Centrality Analysis** — Degree, betweenness, and PageRank to identify key nodes
3. **Community Detection** — Identify coordinated clusters using greedy modularity
4. **CIB Detection** — Coordinated Inauthentic Behavior scoring
5. **Cross-Platform Network Mapping** — Unified graphs spanning Twitter + Telegram
6. **Visualization** — Network plots with role-based coloring and platform markers

In [ ]:
# === Cell 1: Imports and Configuration ===

import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path
from collections import defaultdict

# Project paths
PROJECT_ROOT = Path('.').resolve().parent
FIXTURES_DIR = PROJECT_ROOT / 'tests' / 'fixtures'

# Platform colors
PLATFORM_COLORS = {
    'twitter': '#1DA1F2',
    'telegram': '#0088cc',
    'facebook': '#4267B2',
    'youtube': '#FF0000',
    'default': '#888888',
}

# Role shapes (for node sizing)
ROLE_SIZES = {
    'subject': 1200,
    'source': 1000,
    'amplifier': 600,
    'peer': 800,
    'target': 500,
    'unknown': 400,
}

# Edge type styles
EDGE_COLORS = {
    'retweet': '#2ecc71',
    'forward': '#e67e22',
    'mention': '#95a5a6',
    'amplification': '#e74c3c',
    'reply': '#3498db',
    'quote': '#9b59b6',
    'follow': '#bdc3c7',
}

print(f"NetworkX version: {nx.__version__}")
print(f"Fixtures: {[f.name for f in FIXTURES_DIR.glob('*.json')]}")

## 1. Graph Construction from Briefing Data

Build a directed, weighted graph from the `network_graph` field in a CDDBS briefing.
Each node represents an account/channel with platform and role metadata.
Each edge represents an interaction (retweet, forward, mention, etc.) with a weight.

In [ ]:
# === Cell 2: Graph Builder ===
# Convert CDDBS briefing network_graph JSON to a networkx DiGraph

def build_graph_from_briefing(briefing: dict) -> nx.DiGraph:
    """Build a networkx DiGraph from a CDDBS briefing's network_graph field.
    
    Args:
        briefing: Parsed briefing JSON with network_graph field
    
    Returns:
        Directed weighted graph with node/edge attributes
    """
    G = nx.DiGraph()
    
    network = briefing.get('network_graph', {})
    if not network:
        return G
    
    # Add nodes with attributes
    for node in network.get('nodes', []):
        G.add_node(
            node['id'],
            platform=node.get('platform', 'unknown'),
            role=node.get('role', 'unknown'),
            label=node.get('label', node['id']),
        )
    
    # Add edges with attributes
    for edge in network.get('edges', []):
        G.add_edge(
            edge['from'],
            edge['to'],
            weight=edge.get('weight', 1),
            edge_type=edge.get('type', 'unknown'),
            cross_platform=edge.get('cross_platform', False),
        )
    
    # Store community data as graph attribute
    communities = network.get('communities', [])
    G.graph['communities'] = communities
    
    # Assign community labels to nodes
    for comm in communities:
        for member in comm.get('members', []):
            if member in G:
                G.nodes[member]['community'] = comm.get('id', '')
                G.nodes[member]['community_label'] = comm.get('label', '')
    
    return G


print("build_graph_from_briefing() defined.")

In [ ]:
# === Cell 3: Load All Briefings with Network Data ===
# Load fixtures and build graphs for those containing network_graph

fixture_files = sorted(FIXTURES_DIR.glob('*.json'))
briefings = {}
graphs = {}

for fpath in fixture_files:
    with open(fpath) as f:
        briefing = json.load(f)
    
    briefings[fpath.stem] = briefing
    
    if briefing.get('network_graph', {}).get('nodes'):
        G = build_graph_from_briefing(briefing)
        graphs[fpath.stem] = G
        subject = briefing.get('subject_profile', {}).get('account_handle', 'unknown')
        print(f"  {fpath.stem:35s} -> {G.number_of_nodes()} nodes, {G.number_of_edges()} edges  (subject: {subject})")
    else:
        print(f"  {fpath.stem:35s} -> No network graph data")

print(f"\nGraphs built: {len(graphs)}")

## 2. Centrality Analysis

Compute three centrality measures for each node to identify key actors:
- **Degree centrality** — Most connected nodes (hubs)
- **Betweenness centrality** — Bridge nodes between communities
- **PageRank** — Influence weighted by importance of connections

In [ ]:
# === Cell 4: Centrality Computation ===
# Compute centrality measures for all graphs

def compute_centrality(G: nx.DiGraph) -> dict:
    """Compute centrality measures for all nodes in the graph."""
    results = {}
    
    # Degree centrality (normalized)
    in_degree = nx.in_degree_centrality(G)
    out_degree = nx.out_degree_centrality(G)
    
    # Betweenness centrality
    betweenness = nx.betweenness_centrality(G, weight='weight')
    
    # PageRank
    try:
        pagerank = nx.pagerank(G, weight='weight')
    except nx.PowerIterationFailedConvergence:
        pagerank = {n: 1/G.number_of_nodes() for n in G.nodes()}
    
    for node in G.nodes():
        results[node] = {
            'label': G.nodes[node].get('label', node),
            'platform': G.nodes[node].get('platform', 'unknown'),
            'role': G.nodes[node].get('role', 'unknown'),
            'in_degree': round(in_degree.get(node, 0), 3),
            'out_degree': round(out_degree.get(node, 0), 3),
            'betweenness': round(betweenness.get(node, 0), 3),
            'pagerank': round(pagerank.get(node, 0), 3),
        }
    
    return results


# Run centrality analysis on all graphs
for name, G in graphs.items():
    print(f"\n{'='*70}")
    print(f"Centrality Analysis: {name}")
    print(f"{'='*70}")
    
    centrality = compute_centrality(G)
    
    # Sort by PageRank (most influential first)
    sorted_nodes = sorted(centrality.items(), key=lambda x: -x[1]['pagerank'])
    
    print(f"{'Node':30s} {'Platform':10s} {'Role':12s} {'InDeg':7s} {'OutDeg':7s} {'Between':8s} {'PageRank':8s}")
    print("-" * 85)
    for node_id, metrics in sorted_nodes:
        print(f"  {metrics['label']:28s} {metrics['platform']:10s} {metrics['role']:12s} "
              f"{metrics['in_degree']:6.3f}  {metrics['out_degree']:6.3f}  {metrics['betweenness']:7.3f}  {metrics['pagerank']:7.3f}")
    
    # Identify key nodes
    top_pr = sorted_nodes[0]
    top_btw = max(centrality.items(), key=lambda x: x[1]['betweenness'])
    print(f"\n  Most influential (PageRank):       {top_pr[1]['label']} ({top_pr[1]['pagerank']:.3f})")
    print(f"  Key bridge (Betweenness):           {top_btw[1]['label']} ({top_btw[1]['betweenness']:.3f})")

## 3. Community Detection

We use two approaches:
1. **Pre-labeled communities** from the briefing data (analyst-defined)
2. **Algorithmic detection** using greedy modularity optimization (equivalent to Louvain for small networks)

In [ ]:
# === Cell 5: Community Detection ===
# Compare analyst-defined communities with algorithmic detection

def detect_communities(G: nx.DiGraph) -> list:
    """Detect communities using greedy modularity on the undirected version."""
    # Convert to undirected for community detection
    G_undirected = G.to_undirected()
    
    # Greedy modularity optimization
    communities = list(nx.community.greedy_modularity_communities(G_undirected, weight='weight'))
    
    return [sorted(list(c)) for c in communities]


for name, G in graphs.items():
    print(f"\n{'='*70}")
    print(f"Community Analysis: {name}")
    print(f"{'='*70}")
    
    # 1. Analyst-defined communities (from briefing)
    print("\n  Analyst-Defined Communities:")
    for comm in G.graph.get('communities', []):
        members_str = ', '.join(comm.get('members', []))
        print(f"    [{comm.get('id', '?')}] {comm.get('label', 'Unnamed'):40s} -> {members_str}")
    
    # 2. Algorithmic detection
    detected = detect_communities(G)
    print(f"\n  Algorithmically Detected Communities ({len(detected)} clusters):")
    for i, comm in enumerate(detected):
        labels = [G.nodes[n].get('label', n) for n in comm]
        platforms = set(G.nodes[n].get('platform', '?') for n in comm)
        print(f"    Cluster {i+1} [{','.join(platforms)}]: {', '.join(labels)}")
    
    # 3. Modularity score
    G_undirected = G.to_undirected()
    modularity = nx.community.modularity(G_undirected, [set(c) for c in detected], weight='weight')
    print(f"\n  Modularity Score: {modularity:.3f}")
    if modularity > 0.3:
        print(f"  -> Good community structure (>{0.3})")
    elif modularity > 0.1:
        print(f"  -> Moderate community structure")
    else:
        print(f"  -> Weak community structure")

## 4. Forwarding Chain Analysis (Telegram-Specific)

Analyze the forwarding chain topology in Telegram networks.
Key metrics:
- **Forwarding depth** — How many hops from original source
- **Forwarding breadth** — How many channels forward from the same source
- **Amplification ratio** — Total downstream weight / source weight

In [ ]:
# === Cell 6: Forwarding Chain Analysis ===
# Analyze Telegram forwarding patterns in the network

def analyze_forwarding_chains(G: nx.DiGraph) -> dict:
    """Analyze forwarding chain topology in a CDDBS network graph."""
    
    # Extract forward edges only
    forward_edges = [
        (u, v, d) for u, v, d in G.edges(data=True)
        if d.get('edge_type') == 'forward'
    ]
    
    if not forward_edges:
        return {'has_forwarding': False}
    
    # Build forwarding subgraph
    F = nx.DiGraph()
    for u, v, d in forward_edges:
        F.add_edge(u, v, **d)
        # Copy node attributes
        for n in [u, v]:
            if n in G:
                F.nodes[n].update(G.nodes[n])
    
    # Identify source nodes (nodes with only outgoing forward edges in this context)
    sources = [n for n in F.nodes() if F.in_degree(n) == 0]
    sinks = [n for n in F.nodes() if F.out_degree(n) == 0]
    
    # Compute forwarding metrics
    total_forward_weight = sum(d.get('weight', 1) for _, _, d in forward_edges)
    
    # Longest forwarding chain (path)
    max_depth = 0
    for source in sources:
        for sink in sinks:
            try:
                path_length = nx.shortest_path_length(F, source, sink)
                max_depth = max(max_depth, path_length)
            except nx.NetworkXNoPath:
                continue
    
    # Forwarding breadth (average out-degree of forward sources)
    source_out_degrees = [F.out_degree(s) for s in sources]
    avg_breadth = sum(source_out_degrees) / max(len(source_out_degrees), 1)
    
    # Topology pattern detection
    if len(sources) == 1 and len(sinks) >= 3:
        topology = 'star'  # One source, many amplifiers
    elif max_depth >= 3:
        topology = 'chain'  # Linear forwarding pipeline
    elif len(forward_edges) > len(F.nodes()):
        topology = 'mesh'  # Dense mutual forwarding
    else:
        topology = 'hub_and_spoke'
    
    return {
        'has_forwarding': True,
        'forward_edges': len(forward_edges),
        'total_forward_weight': total_forward_weight,
        'sources': [G.nodes[s].get('label', s) for s in sources],
        'sinks': [G.nodes[s].get('label', s) for s in sinks],
        'max_chain_depth': max_depth,
        'avg_forwarding_breadth': round(avg_breadth, 1),
        'topology': topology,
    }


# Run forwarding chain analysis
for name, G in graphs.items():
    print(f"\n{'='*60}")
    print(f"Forwarding Chain Analysis: {name}")
    print(f"{'='*60}")
    
    fwd = analyze_forwarding_chains(G)
    
    if not fwd['has_forwarding']:
        print("  No forwarding edges in this network.")
        continue
    
    print(f"  Forward edges:         {fwd['forward_edges']}")
    print(f"  Total forward weight:  {fwd['total_forward_weight']}")
    print(f"  Source nodes:          {', '.join(fwd['sources'])}")
    print(f"  Sink nodes:            {', '.join(fwd['sinks'])}")
    print(f"  Max chain depth:       {fwd['max_chain_depth']} hops")
    print(f"  Avg breadth:           {fwd['avg_forwarding_breadth']} channels")
    print(f"  Topology pattern:      {fwd['topology'].upper()}")

## 5. CIB (Coordinated Inauthentic Behavior) Detection

Score the network for coordination signals:
- High edge density within communities
- Cross-platform amplification patterns
- Asymmetric influence flow (few sources → many amplifiers)

In [ ]:
# === Cell 7: CIB Detection Scoring ===
# Score the network for coordinated inauthentic behavior indicators

def score_cib_indicators(G: nx.DiGraph) -> dict:
    """Score a network for coordinated inauthentic behavior (CIB) indicators."""
    
    scores = {}
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    
    if n_nodes < 2:
        return {'total_score': 0, 'assessment': 'Insufficient data'}
    
    # 1. Amplification asymmetry (0-25)
    # Are there few sources feeding many amplifiers?
    roles = defaultdict(list)
    for n, data in G.nodes(data=True):
        roles[data.get('role', 'unknown')].append(n)
    
    n_sources = len(roles.get('source', []))
    n_amplifiers = len(roles.get('amplifier', []))
    n_subjects = len(roles.get('subject', []))
    
    if n_sources > 0 and n_amplifiers > 0:
        amp_ratio = n_amplifiers / n_sources
        scores['amplification_asymmetry'] = min(amp_ratio * 8, 25)
    else:
        scores['amplification_asymmetry'] = 0
    
    # 2. Edge density within communities (0-25)
    communities = G.graph.get('communities', [])
    if communities:
        cohesions = []
        for comm in communities:
            members = [m for m in comm.get('members', []) if m in G]
            if len(members) >= 2:
                subgraph = G.subgraph(members)
                possible_edges = len(members) * (len(members) - 1)
                actual_edges = subgraph.number_of_edges()
                density = actual_edges / possible_edges if possible_edges > 0 else 0
                cohesions.append(density)
        
        avg_cohesion = sum(cohesions) / max(len(cohesions), 1)
        scores['community_cohesion'] = avg_cohesion * 25
    else:
        scores['community_cohesion'] = 0
    
    # 3. Cross-platform coordination (0-25)
    xp_edges = sum(1 for _, _, d in G.edges(data=True) if d.get('cross_platform', False))
    platforms = set(d.get('platform', '') for _, d in G.nodes(data=True))
    
    if len(platforms) > 1 and xp_edges > 0:
        scores['cross_platform'] = min(xp_edges * 12, 25)
    elif len(platforms) > 1:
        scores['cross_platform'] = 10  # Multi-platform but no explicit XP edges
    else:
        scores['cross_platform'] = 0
    
    # 4. High-weight concentration (0-25)
    # Is there disproportionate weight on few edges?
    weights = [d.get('weight', 1) for _, _, d in G.edges(data=True)]
    if weights:
        max_weight = max(weights)
        mean_weight = sum(weights) / len(weights)
        concentration = max_weight / max(mean_weight, 1)
        scores['weight_concentration'] = min(concentration * 5, 25)
    else:
        scores['weight_concentration'] = 0
    
    # Total score
    total = sum(scores.values())
    
    # Assessment
    if total >= 70:
        assessment = 'HIGH — Strong indicators of coordinated inauthentic behavior'
    elif total >= 45:
        assessment = 'MODERATE — Some coordination indicators present'
    elif total >= 20:
        assessment = 'LOW — Minimal coordination indicators'
    else:
        assessment = 'NONE — No significant coordination detected'
    
    return {
        'scores': {k: round(v, 1) for k, v in scores.items()},
        'total_score': round(total, 1),
        'max_score': 100,
        'assessment': assessment,
        'network_stats': {
            'nodes': n_nodes,
            'edges': n_edges,
            'platforms': sorted(platforms),
            'roles': {k: len(v) for k, v in roles.items()},
        }
    }


# Run CIB scoring on all networks
for name, G in graphs.items():
    print(f"\n{'='*65}")
    print(f"CIB Assessment: {name}")
    print(f"{'='*65}")
    
    cib = score_cib_indicators(G)
    
    print(f"\n  Network: {cib['network_stats']['nodes']} nodes, {cib['network_stats']['edges']} edges")
    print(f"  Platforms: {', '.join(cib['network_stats']['platforms'])}")
    print(f"  Roles: {cib['network_stats']['roles']}")
    print(f"")
    print(f"  CIB Score Breakdown:")
    for indicator, score in cib['scores'].items():
        bar = '#' * int(score) + '.' * (25 - int(score))
        print(f"    {indicator:30s}  [{bar}] {score:5.1f}/25")
    print(f"")
    print(f"  TOTAL: {cib['total_score']:.1f}/{cib['max_score']}")
    print(f"  Assessment: {cib['assessment']}")

## 6. Network Visualization

Visualize networks with:
- Node color by platform (Twitter=blue, Telegram=teal)
- Node size by PageRank influence
- Edge color by interaction type
- Edge thickness by weight
- Community grouping with labels

In [ ]:
# === Cell 8: Network Visualization ===
# Render network graphs with role/platform-based styling

def visualize_network(G: nx.DiGraph, title: str = 'CDDBS Network Graph'):
    """Visualize a CDDBS network graph with platform and role styling."""
    
    if G.number_of_nodes() == 0:
        print("Empty graph — nothing to visualize.")
        return
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    
    # Layout
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42, weight='weight')
    
    # Node colors by platform
    node_colors = [
        PLATFORM_COLORS.get(G.nodes[n].get('platform', 'default'), PLATFORM_COLORS['default'])
        for n in G.nodes()
    ]
    
    # Node sizes by role + PageRank influence
    pagerank = nx.pagerank(G, weight='weight')
    node_sizes = [
        ROLE_SIZES.get(G.nodes[n].get('role', 'unknown'), 400) * (0.5 + pagerank.get(n, 0) * 5)
        for n in G.nodes()
    ]
    
    # Edge colors by type
    edge_colors = [
        EDGE_COLORS.get(G.edges[e].get('edge_type', 'unknown'), '#cccccc')
        for e in G.edges()
    ]
    
    # Edge widths by weight (normalized)
    weights = [G.edges[e].get('weight', 1) for e in G.edges()]
    max_w = max(weights) if weights else 1
    edge_widths = [1 + (w / max_w) * 4 for w in weights]
    
    # Draw edges
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edge_color=edge_colors,
        width=edge_widths,
        alpha=0.6,
        arrows=True,
        arrowsize=15,
        connectionstyle='arc3,rad=0.1',
    )
    
    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=node_colors,
        node_size=node_sizes,
        alpha=0.9,
        edgecolors='white',
        linewidths=2,
    )
    
    # Labels
    labels = {n: G.nodes[n].get('label', n) for n in G.nodes()}
    nx.draw_networkx_labels(
        G, pos, labels, ax=ax,
        font_size=8,
        font_weight='bold',
    )
    
    # Edge weight labels
    edge_labels = {(u, v): f"{d.get('weight', '')}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels, ax=ax,
        font_size=7,
        font_color='gray',
    )
    
    # Legend for platforms
    platform_patches = [
        mpatches.Patch(color=color, label=platform.title())
        for platform, color in PLATFORM_COLORS.items()
        if platform != 'default' and any(
            G.nodes[n].get('platform') == platform for n in G.nodes()
        )
    ]
    
    # Legend for edge types
    edge_type_set = set(G.edges[e].get('edge_type', '') for e in G.edges())
    edge_patches = [
        mpatches.Patch(color=EDGE_COLORS.get(et, '#ccc'), label=f"Edge: {et}")
        for et in sorted(edge_type_set) if et
    ]
    
    ax.legend(handles=platform_patches + edge_patches, loc='upper left', fontsize=8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()


print("visualize_network() defined.")

In [ ]:
# === Cell 9: Visualize All Networks ===
# Generate visualizations for each briefing with network data

for name, G in graphs.items():
    subject = briefings[name].get('subject_profile', {}).get('account_handle', 'unknown')
    title = f"CDDBS Network: {subject}\n({name})"
    visualize_network(G, title)

## 7. Unified Cross-Platform Network

Merge all individual network graphs into a single unified graph
to see the full information ecosystem across briefings.

In [ ]:
# === Cell 10: Unified Cross-Platform Graph ===
# Merge all individual briefing networks into one unified graph

unified = nx.DiGraph()

for name, G in graphs.items():
    # Add all nodes (update attributes if node already exists)
    for n, data in G.nodes(data=True):
        if n in unified:
            unified.nodes[n].update(data)
        else:
            unified.add_node(n, **data)
    
    # Add/merge edges (sum weights for duplicate edges)
    for u, v, data in G.edges(data=True):
        if unified.has_edge(u, v):
            unified.edges[u, v]['weight'] += data.get('weight', 1)
        else:
            unified.add_edge(u, v, **data)

print(f"Unified Network Statistics:")
print(f"  Total nodes:     {unified.number_of_nodes()}")
print(f"  Total edges:     {unified.number_of_edges()}")
print(f"  Platforms:       {sorted(set(d.get('platform', '?') for _, d in unified.nodes(data=True)))}")
print(f"  Connected:       {nx.is_weakly_connected(unified)}")
print(f"  Components:      {nx.number_weakly_connected_components(unified)}")

# Visualize unified graph
visualize_network(unified, 'CDDBS Unified Cross-Platform Network\n(All Briefings Combined)')

In [ ]:
# === Cell 11: Platform Diversification Index ===
# Compute PDI for each node to measure cross-platform reach
# PDI = 1 - sum(share_per_platform^2) — Herfindahl-Hirschman inverse

def compute_platform_diversification(G: nx.DiGraph) -> dict:
    """Compute Platform Diversification Index for each node."""
    results = {}
    
    for node in G.nodes():
        # Count interactions by platform of connected nodes
        platform_weights = defaultdict(float)
        total_weight = 0
        
        for _, target, data in G.out_edges(node, data=True):
            p = G.nodes[target].get('platform', 'unknown')
            w = data.get('weight', 1)
            platform_weights[p] += w
            total_weight += w
        
        for source, _, data in G.in_edges(node, data=True):
            p = G.nodes[source].get('platform', 'unknown')
            w = data.get('weight', 1)
            platform_weights[p] += w
            total_weight += w
        
        if total_weight == 0:
            results[node] = {'pdi': 0.0, 'platforms': {}}
            continue
        
        shares = {p: w / total_weight for p, w in platform_weights.items()}
        hhi = sum(s ** 2 for s in shares.values())
        pdi = 1 - hhi  # PDI: 0 = single platform, 1 = perfectly diversified
        
        results[node] = {
            'pdi': round(pdi, 3),
            'platforms': {p: round(s, 3) for p, s in shares.items()},
        }
    
    return results


pdi_results = compute_platform_diversification(unified)

print("Platform Diversification Index (PDI)")
print("=" * 70)
print(f"{'Node':30s} {'PDI':6s}  Platform Distribution")
print("-" * 70)

for node, data in sorted(pdi_results.items(), key=lambda x: -x[1]['pdi']):
    label = unified.nodes[node].get('label', node)
    dist = ', '.join(f"{p}:{s:.0%}" for p, s in data['platforms'].items())
    print(f"  {label:28s} {data['pdi']:.3f}   {dist}")

## 8. Conclusions

### Key Results
1. **Graph construction** from briefing `network_graph` data works correctly — nodes, edges, and communities are preserved
2. **Centrality analysis** identifies RT English and state media channels as highest-influence nodes (by PageRank)
3. **Community detection** algorithmically separates state media, proxy operations, and amplifier clusters
4. **Forwarding chain analysis** reveals star and hub-and-spoke topologies typical of state media propagation
5. **CIB scoring** provides a quantitative assessment of coordination indicators
6. **Cross-platform PDI** measures how effectively entities span multiple platforms

### Production Readiness
- Graph construction and centrality analysis are ready for production integration
- Community detection should use Louvain (requires `python-louvain` package) for larger networks
- CIB scoring thresholds need calibration with more diverse network examples
- Visualization renders well for networks up to ~50 nodes; larger networks need D3.js/Cytoscape.js

### Next Steps
- Integrate with platform adapters for automated graph construction from API data
- Add temporal evolution analysis (how networks change over time)
- Implement Infomap for directed flow-based community detection